In [1]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/clean_modules_fxns.R")

In [2]:
mod_def <- "TopModPosBC"

In [3]:
marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

## Prep data

In [4]:
data_source <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned"

In [5]:
bulk_expr <- fread("data/cleaned/ModSeed/GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned.csv", data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

In [6]:
network_dir <- "GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_mergeParam0.95_subsetCutoff2.544_Modules"

networks <- list.files(network_dir, pattern="signum", full.names=TRUE)
networks <- networks[lengths(lapply(networks, list.files)) > 0]

In [7]:
networks

[1] "GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_mergeParam0.95_subsetCutoff2.544_Modules/Bicor-None_signum0.454_minSize6_merge_ME_0.95_18750"

In [ ]:
# Manually picking "noise" modules to remove:

mods <- c(
    "tan1", "aliceblue", "lavenderblush", "grey18", "gray47", 
    "firebrick1", "navy", "lavenderblush1", "olivedrab1", "grey39", 
    "orchid3", "grey44", "chocolate4", "cadetblue2", "gray42", 
    "deepskyblue3", "grey32", "dimgray", "mediumspringgreen", 
    "lightsteelblue", "deepskyblue1", "darkgrey", "springgreen4",
    "sienna2", "lightgoldenrod4", "gray13", "yellow", "gray2", 
    "chocolate1", "gray72", "gray39", "grey38"
)

i <- 1
kME <- fread(list.files(networks[i], pattern="kME", full.names=TRUE), data.table=FALSE)
modEigs <- fread(list.files(networks[i], pattern="eigen", full.names=TRUE), data.table=FALSE)
mod_genes <- unlist(lapply(mods, function(module) {
        get_mod_genes(kME, module, mod_def, n_genes=NULL)
    }))

In [10]:
length(mod_genes)

[1] 3005

In [11]:
exclude <- unique(unlist(marker_genes_list)) # Don't remove genes associated with cell types of interest

mod_genes <- mod_genes[!mod_genes %in% exclude]

expr_filtered <- bulk_expr[!bulk_expr[,1] %in% mod_genes,] 

## Save

In [12]:
outdir <- paste0("data/cleaned/", mod_def)

if (!dir.exists(outdir)) {
    dir.create(outdir, recursive=TRUE)
}

filename <- paste0(outdir, "/", data_source, "_", nrow(expr_filtered), "genes_cleaned.csv")

fwrite(expr_filtered, file=filename)